# Stage 6 local LLM baseline

Run Qwen3-8B locally in Colab and generate Vega-Lite JSON predictions.

In [5]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import quote, urlsplit, urlunsplit

from google.colab import drive

try:
    from google.colab import userdata
except Exception:
    userdata = None

DRIVE_ROOT = Path('/content/drive/MyDrive/diploma/petr_text_to_visualization_part')
REPO_ROOT = Path('/content/petukhov_t2v_repo')
REPO_URL = 'https://github.com/petrussia/NL2BI-AI-assistant.git'
BRANCH = 'experiments/peter'
TOKEN_FILE = DRIVE_ROOT / 'secrets' / 'github_token.txt'

def read_token() -> str | None:
    if userdata is not None:
        try:
            token = userdata.get('GITHUB_TOKEN')
            if token:
                return token.strip()
        except Exception:
            pass
    if TOKEN_FILE.exists():
        return TOKEN_FILE.read_text(encoding='utf-8').strip()
    return None

def auth_url(token: str | None) -> str:
    if not token:
        return REPO_URL
    parts = urlsplit(REPO_URL)
    return urlunsplit((parts.scheme, f'x-access-token:{quote(token, safe="")}@{parts.netloc}', parts.path, parts.query, parts.fragment))

def run(cmd: list[str], cwd: Path | None = None) -> None:
    shown = ['***' if 'x-access-token:' in part else part for part in cmd]
    print('RUN', ' '.join(shown))
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['T2V_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ['T2V_REPO_ROOT'] = str(REPO_ROOT)

repo_url = auth_url(read_token())
if not REPO_ROOT.exists():
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', repo_url, str(REPO_ROOT)])
else:
    run(['git', 'remote', 'set-url', 'origin', repo_url], cwd=REPO_ROOT)
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_ROOT)
    run(['git', 'checkout', BRANCH], cwd=REPO_ROOT)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_ROOT)
run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_ROOT)
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt')])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_ROOT)])
run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.51', 'accelerate>=0.30', 'bitsandbytes', 'sentencepiece', 'safetensors'])
run([sys.executable, '-c', 'import t2v_eval; print(t2v_eval.__version__)'])
run([sys.executable, '-c', 'from t2v_eval.baselines.llm_vegalite import LLMVegaLiteConfig; print(LLMVegaLiteConfig().model_id)'])
print('STAGE6_SETUP_OK')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RUN git remote set-url origin ***
RUN git fetch origin experiments/peter
RUN git checkout experiments/peter
RUN git pull --ff-only origin experiments/peter
RUN git remote set-url origin https://github.com/petrussia/NL2BI-AI-assistant.git
RUN /usr/bin/python3 -m pip install -q -r /content/petukhov_t2v_repo/requirements.txt
RUN /usr/bin/python3 -m pip install -q -e /content/petukhov_t2v_repo
RUN /usr/bin/python3 -m pip install -q transformers>=4.51 accelerate>=0.30 bitsandbytes sentencepiece safetensors
RUN /usr/bin/python3 -c import t2v_eval; print(t2v_eval.__version__)
RUN /usr/bin/python3 -c from t2v_eval.baselines.llm_vegalite import LLMVegaLiteConfig; print(LLMVegaLiteConfig().model_id)
STAGE6_SETUP_OK


In [6]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

drive_root = Path(os.environ['T2V_DRIVE_ROOT'])
repo_root = Path(os.environ['T2V_REPO_ROOT'])
examples = drive_root / 'datasets' / 'processed' / 'nvbench_postquery' / 'examples_sample200.jsonl'
cmd = [
    sys.executable,
    str(repo_root / 'scripts' / 'run_llm_experiment.py'),
    '--examples', str(examples),
    '--drive-root', str(drive_root),
    '--run-id', 'stage6_qwen3_8b_fast_latency2',
    '--sample-size', '2',
    '--model-id', 'Qwen/Qwen3-8B',
    '--quantization', '4bit',
    '--max-new-tokens', '384',
    '--temperature', '0.0',
    '--top-p', '1.0',
    '--sample-rows', '5',
    '--render-limit', '0',
    '--json',
]
print('RUN', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'latency2 LLM run failed: {result.returncode}')
print('STAGE6_LATENCY2_OK')


RUN /usr/bin/python3 /content/petukhov_t2v_repo/scripts/run_llm_experiment.py --examples /content/drive/MyDrive/diploma/petr_text_to_visualization_part/datasets/processed/nvbench_postquery/examples_sample200.jsonl --drive-root /content/drive/MyDrive/diploma/petr_text_to_visualization_part --run-id stage6_qwen3_8b_fast_latency2 --sample-size 2 --model-id Qwen/Qwen3-8B --quantization 4bit --max-new-tokens 384 --temperature 0.0 --top-p 1.0 --sample-rows 5 --render-limit 0 --json
{
  "run_id": "stage6_qwen3_8b_fast_latency2",
  "examples_path": "/content/drive/MyDrive/diploma/petr_text_to_visualization_part/datasets/processed/nvbench_postquery/examples_sample200.jsonl",
  "examples_used": "/content/drive/MyDrive/diploma/petr_text_to_visualization_part/runs/stage6_qwen3_8b_fast_latency2/examples_used.jsonl",
  "drive_root": "/content/drive/MyDrive/diploma/petr_text_to_visualization_part",
  "sample_size": 2,
  "method": "B3_local_llm_qwen3_8b",
  "llm_config": {
    "model_id": "Qwen/Qwen3-

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

drive_root = Path(os.environ['T2V_DRIVE_ROOT'])
repo_root = Path(os.environ['T2V_REPO_ROOT'])
examples = drive_root / 'datasets' / 'processed' / 'nvbench_postquery' / 'examples_sample200.jsonl'
cmd = [
    sys.executable,
    str(repo_root / 'scripts' / 'run_llm_experiment.py'),
    '--examples', str(examples),
    '--drive-root', str(drive_root),
    '--run-id', 'stage6_qwen3_8b_fast_smoke5',
    '--sample-size', '5',
    '--model-id', 'Qwen/Qwen3-8B',
    '--quantization', '4bit',
    '--max-new-tokens', '384',
    '--temperature', '0.0',
    '--top-p', '1.0',
    '--sample-rows', '5',
    '--render-limit', '0',
    '--json',
]
print('RUN', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'smoke5 LLM run failed: {result.returncode}')
print('STAGE6_SMOKE5_OK')


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

drive_root = Path(os.environ['T2V_DRIVE_ROOT'])
repo_root = Path(os.environ['T2V_REPO_ROOT'])
examples = drive_root / 'datasets' / 'processed' / 'nvbench_postquery' / 'examples_sample200.jsonl'
cmd = [
    sys.executable,
    str(repo_root / 'scripts' / 'run_llm_experiment.py'),
    '--examples', str(examples),
    '--drive-root', str(drive_root),
    '--run-id', 'stage6_qwen3_8b_fast_sample20',
    '--sample-size', '20',
    '--model-id', 'Qwen/Qwen3-8B',
    '--quantization', '4bit',
    '--max-new-tokens', '384',
    '--temperature', '0.0',
    '--top-p', '1.0',
    '--sample-rows', '5',
    '--render-limit', '20',
    '--json',
]
print('RUN', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'sample20 LLM run failed: {result.returncode}')
print('STAGE6_SAMPLE20_OK')


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

drive_root = Path(os.environ['T2V_DRIVE_ROOT'])
repo_root = Path(os.environ['T2V_REPO_ROOT'])
examples = drive_root / 'datasets' / 'processed' / 'nvbench_postquery' / 'examples_sample200.jsonl'
cmd = [
    sys.executable,
    str(repo_root / 'scripts' / 'run_llm_experiment.py'),
    '--examples', str(examples),
    '--drive-root', str(drive_root),
    '--run-id', 'stage6_qwen3_8b_fast_sample50',
    '--sample-size', '50',
    '--model-id', 'Qwen/Qwen3-8B',
    '--quantization', '4bit',
    '--max-new-tokens', '384',
    '--temperature', '0.0',
    '--top-p', '1.0',
    '--sample-rows', '5',
    '--render-limit', '20',
    '--json',
]
print('RUN', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'sample50 LLM run failed: {result.returncode}')
print('STAGE6_SAMPLE50_OK')


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

drive_root = Path(os.environ['T2V_DRIVE_ROOT'])
run_id = os.environ.get('STAGE6_FINAL_RUN_ID', 'stage6_qwen3_8b_fast_sample20')
run_root = drive_root / 'runs' / run_id
method = 'B3_local_llm_qwen3_8b'
required = [
    run_root / 'examples_used.jsonl',
    run_root / 'predictions' / f'{method}.jsonl',
    run_root / 'metrics' / method / 'aggregate_metrics.csv',
    run_root / 'metrics' / method / 'per_example_metrics.csv',
    run_root / 'metrics' / method / 'evaluation_summary.json',
    run_root / 'experiment_summary.json',
    run_root / 'runtime_info.json',
    run_root / 'llm_config.json',
    run_root / 'gpu_runtime_before.json',
    run_root / 'gpu_runtime_after.json',
    run_root / 'pip_freeze.txt',
]
missing = [str(path) for path in required if not path.exists()]
rendered = len(list((run_root / 'rendered' / method).glob('*.png'))) if (run_root / 'rendered' / method).exists() else 0
summary = json.loads((run_root / 'experiment_summary.json').read_text(encoding='utf-8')) if (run_root / 'experiment_summary.json').exists() else {}
print(json.dumps({'run_root': str(run_root), 'missing': missing, 'rendered': rendered, 'summary': summary}, ensure_ascii=False, indent=2))
if missing:
    raise RuntimeError('Missing Stage 6 artifacts')
print('STAGE6_ARTIFACTS_OK')
